# Chess Engine with TensorFlow

## Dataset

In [1]:
import os
# inspired by Github user Skripkon 
# https://github.com/Skripkon/chess-engine/blob/main/engines/tensorflow/train_and_predict.ipynb
# used to train models from scratch
# took around 3 hours for 20000 games

# get the game files
files = [file for file in os.listdir("simulated_games_filtered_PGN") if file.endswith(".pgn")]

In [2]:
from chess import pgn

# load the games
def load_pgn(file_path):
    games = []
    with open(file_path, 'r') as pgn_file:
        while True:
            game = pgn.read_game(pgn_file)
            if game is None:
                break
            games.append(game)
            
    return games

In [3]:
from tqdm import tqdm
# write all the games together 

games = []
for file in tqdm(files):
    games.extend(load_pgn(f"simulated_games_filtered_PGN/{file}"))

100%|██████████| 11/11 [00:29<00:00,  2.65s/it]


In [4]:
len(games) # check how many 

10085

## Build & train a neural network

In [5]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from chess import Board
import tensorflow as tf

In [ ]:
#translating the board into a matrix for the predition process 
def board_to_matrix(board: Board):
    matrix = np.zeros((8, 8, 12))
    piece_map = board.piece_map()
    for square, piece in piece_map.items():
        row, col = divmod(square, 8)
        piece_type = piece.piece_type - 1
        piece_color = 0 if piece.color else 6
        matrix[row, col, piece_type + piece_color] = 1
    return matrix

# create the inputs based off game position and next move 
def create_input_for_nn(games):
    X = []
    y = []
    for game in games:
        board = game.board()
        for move in game.mainline_moves():
            X.append(board_to_matrix(board))
            y.append(move.uci())
            board.push(move)
    return X, y

# encode all the moves
def encode_moves(moves):
    move_to_int = {move: idx for idx, move in enumerate(set(moves))}
    return [move_to_int[move] for move in moves], move_to_int

In [9]:
# create the training data
X, y = create_input_for_nn(games)
y, move_to_int = encode_moves(y)
y = tf.keras.utils.to_categorical(y, num_classes=len(move_to_int))
X = np.array(X)

{'d3b1': 0, 'd6e8': 1, 'c8d7': 2, 'b2c4': 3, 'g7e5': 4, 'b7b3': 5, 'd4e3': 6, 'b2c1q': 7, 'b6d4': 8, 'e8e1': 9, 'c5d4': 10, 'f1f3': 11, 'd8d1': 12, 'g4g1': 13, 'c3c6': 14, 'g3e1': 15, 'f1f8': 16, 'c7c6': 17, 'b1g1': 18, 'h1h2': 19, 'a2e2': 20, 'f6h6': 21, 'a2c1': 22, 'e3a3': 23, 'g2c6': 24, 'h5c5': 25, 'a2a1r': 26, 'h5g5': 27, 'd2b2': 28, 'c4d4': 29, 'c7h2': 30, 'b4a6': 31, 'g8d8': 32, 'g3f2': 33, 'b6c5': 34, 'f3d1': 35, 'f4h4': 36, 'e4d4': 37, 'c7c1': 38, 'b7a5': 39, 'g8c4': 40, 'a5e5': 41, 'c7f4': 42, 'a4a3': 43, 'a8h8': 44, 'f5c2': 45, 'g2h2': 46, 'g2f1q': 47, 'a2a8': 48, 'h2h1b': 49, 'f5h4': 50, 'a7e3': 51, 'd7h3': 52, 'b4a5': 53, 'c6f6': 54, 'b3a3': 55, 'b4c2': 56, 'h6g8': 57, 'a5c3': 58, 'c6d5': 59, 'b4d3': 60, 'f6f5': 61, 'f4g3': 62, 'd6d7': 63, 'g5f7': 64, 'e5c5': 65, 'f5c8': 66, 'a1g7': 67, 'g8h6': 68, 'b5c5': 69, 'b5c7': 70, 'a7c6': 71, 'd5f6': 72, 'g3a3': 73, 'e2g1': 74, 'd2c1b': 75, 'c7e8': 76, 'e6f5': 77, 'c2c3': 78, 'e4e3': 79, 'c4g4': 80, 'g4g8': 81, 'c3g3': 82, 'b3c2': 

In [10]:
print(len(move_to_int))

1945


In [8]:

# train the model
model = tf.keras.models.Sequential([
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu', input_shape=(8, 8, 12)),
    tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dense(len(move_to_int), activation='softmax')
])
model.compile(optimizer = tf.keras.optimizers.Adam(), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()
model.fit(X, y, epochs=50, validation_split=0.1, batch_size=64)
# save the model
model.save("simulated_filtered_model(original)/SSMF_50EPOCHS.keras")

import pickle
# save the encoding
with open("simulated_filtered_model(original)/move_to_int.pkl", "wb") as f:
    pickle.dump(move_to_int, f)
int_to_move = {v: k for k, v in move_to_int.items()}
with open("simulated_filtered_model(original)/int_to_move.pkl", "wb") as f:
    pickle.dump(int_to_move, f)
# configuration 
config = {
    "epochs": 50,
    "batch_size": 64,
    "validation_split": 0.1,
    "optimizer": "Adam",
    "input_shape": (8, 8, 12),
}
# save the configurations
with open("simulated_filtered_model(original)/train_config.json", "w") as f:
    import json
    json.dump(config, f, indent=4)



Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 6, 6, 64)       │         6,976 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 4, 4, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1945)           │       499,865 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,105,241 (4.22 MB)

 Trainable params: 1,105,241 (4.22 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
15743/15743 ━━━━━━━━━━━━━━━━━━━━ 54s 3ms/step - accuracy: 0.0624 - loss: 5.3353 - val_accuracy: 0.0789 - val_loss: 4.7490
Epoch 2/50
15743/15743 ━━━━━━━━━━━━━━━━━━━━ 49s 3ms/step - accuracy: 0.0864 - loss: 4.5241 - val_accuracy: 0.0867 - val_loss: 4.5160
Epoch 3/50
15743/15743 ━━━━━━━━━━━━━━━━━━━━ 49s 3ms/step - accuracy: 0.0962 - loss: 4.3295 - val_accuracy: 0.0901 - val_loss: 4.4441
Epoch 4/50
15743/15743 ━━━━━━━━━━━━━━━━━━━━ 47s 3ms/step - accuracy: 0.1029 - loss: 4.2256 - val_accuracy: 0.0911 - val_loss: 4.3969
Epoch 5/50
15743/15743 ━━━━━━━━━━━━━━━━━━━━ 51s 3ms/step - accuracy: 0.1072 - loss: 4.1584 - val_accuracy: 0.0943 - val_loss: 4.3675
Epoch 6/50
15743/15743 ━━━━━━━━━━━━━━━━━━━━ 50s 3ms/step - accuracy: 0.1111 - loss: 4.1100 - val_accuracy: 0.0947 - val_loss: 4.3612
Epoch 7/50
15743/15743 ━━━━━━━━━━━━━━━━━━━━ 52s 3ms/step - accuracy: 0.1138 - loss: 4.0714 - val_accuracy: 0.0968 - val_loss: 4.3558
Epoch 8/50
15743/15743 ━━━━━━━━━━━━━━━━━━━━ 53s 3ms/step - accuracy: 